# EDA Notebook 3 v3 — Sentiment140

**Thesis:** *An Ethical Risk Assessment of Sentiment Analysis Models in Social Media Monitoring with a Focus on Bias, Inclusivity, and Misclassification Patterns*  
**Student:** Abhishek Tomar | PN1196973 | LJMU

## Purpose
Sentiment140 is retained as an **external binary robustness dataset**. It is not used to train, tune, select or rank the TweetEval models.

This notebook:
1. loads the full 1.6M-row corpus;
2. audits missing/empty text and duplicate/conflicting normalized texts;
3. audits residual label-source-like emoticons;
4. constructs a deterministic, class-balanced **100,000-row unique-text evaluation sample** after excluding conflicting duplicate labels;
5. preserves raw text for frozen linguistic tagging while creating a separate leakage-controlled `text_eval` for later model evaluation;
6. loads and applies the frozen TweetEval tagging specification without modification;
7. reports pre-priority overlaps and final subgroup coverage;
8. records binary label-space shift explicitly;
9. saves authoritative outputs to `Results/Sentiment140/`.

**Important:** subgroup scarcity never triggers threshold adjustment. Sentiment140 cannot be pooled with TweetEval to manufacture subgroup eligibility.


## Cell 1 — Install and import libraries


In [1]:
!pip install -q datasets emoji pyarrow

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import glob, json, re
import numpy as np
import pandas as pd
import emoji
from datasets import load_dataset

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)
print('Libraries loaded.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 11.1 MB/s eta 0:00:00
Mounted at /content/drive
Libraries loaded.


## Cell 2 — Locate and validate the frozen TweetEval tagging specification


In [2]:
CONFIG_NAME = 'tweeteval_freeze_config.json'
matches = glob.glob(f'/content/drive/MyDrive/**/{CONFIG_NAME}', recursive=True)
if not matches:
    raise FileNotFoundError(f'{CONFIG_NAME} not found. Run and freeze EDA_01 first.')
preferred = [p for p in matches if '/Results/TweetEval/' in p.replace('\\','/')]
config_path = Path(preferred[0] if preferred else matches[0])
with open(config_path, 'r', encoding='utf-8') as f:
    freeze = json.load(f)

assert freeze['emoji_heavy']['operator'] == '>' and float(freeze['emoji_heavy']['threshold']) == 0.05
assert freeze['slang_heavy']['operator'] == '>' and float(freeze['slang_heavy']['threshold']) == 0.10
assert int(freeze['slang_heavy']['lexicon_size']) == 51
assert int(freeze['reference_unmarked']['minimum_word_count']) == 8
assert int(freeze['minimum_n_for_stable_quantitative_interpretation']) == 30
assert freeze['priority_rule'] == ['sarcasm-indicated','emoji-heavy','slang-heavy','reference-unmarked','other']

RESULTS_ROOT = config_path.parent.parent
OUTDIR = RESULTS_ROOT / 'Sentiment140'
OUTDIR.mkdir(parents=True, exist_ok=True)
print('Freeze config:', config_path)
print('Output directory:', OUTDIR)


Freeze config: /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Final_Thesis/EDA/Results/TweetEval/tweeteval_freeze_config.json
Output directory: /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Final_Thesis/EDA/Results/Sentiment140


## Cell 3 — Load Sentiment140 and standardise binary labels


In [3]:
DATASET_REPO = 'contemmcm/sentiment140'
ds = load_dataset(DATASET_REPO)
if 'complete' not in ds:
    raise KeyError(f"Expected split 'complete'; found {list(ds.keys())}")
df = pd.DataFrame(ds['complete'])

# The current Hugging Face representation uses 0=negative, 1=positive.
label_map = {0:'negative', 1:'positive'}
df['sentiment'] = df['label'].map(label_map)
if df['sentiment'].isna().any():
    raise ValueError('Unexpected Sentiment140 label value encountered.')
df['text'] = df['text'].fillna('').astype(str)

print('Rows:', f'{len(df):,}')
print('Columns:', list(df.columns))
print(df['sentiment'].value_counts().to_string())


README.md:   0%|          | 0.00/361 [00:00<?, ?B/s]

polarity.csv: reconstructing file:   0%|          |  0.00B /  124MB            

polarity.csv: downloading bytes:           |  0.00B            

Generating complete split: 0 examples [00:00, ? examples/s]

Rows: 1,600,000
Columns: ['text', 'label', 'sentiment']
sentiment
positive    800000
negative    800000


## Cell 4 — Full-corpus integrity and duplicate-label audit


In [4]:
def normalise_text(text):
    text = str(text).strip().lower()
    return re.sub(r'\s+', ' ', text)

df['text_norm'] = df['text'].map(normalise_text)
missing_text = int(df['text'].isna().sum())
empty_text = int((df['text'].str.strip() == '').sum())

# Group normalized duplicate texts once; this is more informative than counting duplicate rows alone.
dup_audit = (df.groupby('text_norm', observed=True)
    .agg(row_count=('text','size'),
         label_count=('sentiment','nunique'),
         labels=('sentiment', lambda x: '|'.join(sorted(set(map(str,x))))),
         example_text=('text','first'))
    .reset_index())
dup_groups = dup_audit[dup_audit['row_count'] > 1].copy()
conflict_groups = dup_groups[dup_groups['label_count'] > 1].copy()

print('Missing text:', missing_text)
print('Empty text:', empty_text)
print('Unique normalized texts:', f"{df['text_norm'].nunique():,}")
print('Duplicate normalized-text groups:', f'{len(dup_groups):,}')
print('Extra duplicate rows:', f"{int((dup_groups['row_count']-1).sum()):,}")
print('Conflicting-label duplicate groups:', f'{len(conflict_groups):,}')


Missing text: 0
Empty text: 0
Unique normalized texts: 1,576,447
Duplicate normalized-text groups: 9,794
Extra duplicate rows: 23,553
Conflicting-label duplicate groups: 2,385


## Cell 5 — Residual label-source-like emoticon audit


In [6]:
# Sentiment140 was created by distant supervision using emoticons. This audit uses a
# conservative set of common positive/negative ASCII emoticon forms to detect residual
# label-revealing cues. We call them "label-source-like" because this notebook does not
# claim that this regex reproduces the historical collection query exactly.
POS_EMOTICON_RE = re.compile(r'(?:(?::|=|;)-?[)DＰpP])')
NEG_EMOTICON_RE = re.compile(r'(?:(?::|=)-?[(])')
ANY_SOURCE_LIKE_RE = re.compile(r'(?:(?::|=|;)-?[)DＰpP]|(?::|=)-?[(])')

df['has_pos_source_like_emoticon'] = df['text'].str.contains(POS_EMOTICON_RE, regex=True)
df['has_neg_source_like_emoticon'] = df['text'].str.contains(NEG_EMOTICON_RE, regex=True)
df['has_source_like_emoticon'] = df['has_pos_source_like_emoticon'] | df['has_neg_source_like_emoticon']

leakage_summary = pd.DataFrame({
    'measure':['positive-source-like','negative-source-like','any-source-like'],
    'count':[int(df['has_pos_source_like_emoticon'].sum()), int(df['has_neg_source_like_emoticon'].sum()), int(df['has_source_like_emoticon'].sum())]
})
leakage_summary['percentage'] = leakage_summary['count'] / len(df) * 100
print(leakage_summary.to_string(index=False))

print('By class:')
print(pd.crosstab(df['sentiment'], df['has_source_like_emoticon']).to_string())


             measure  count  percentage
positive-source-like  13708    0.856750
negative-source-like    602    0.037625
     any-source-like  14299    0.893688
By class:
has_source_like_emoticon   False  True 
sentiment                              
negative                  794735   5265
positive                  790966   9034


## Cell 6 — Construct deterministic external-evaluation sample


In [7]:
# Do not let repeated text dominate external metrics. Remove conflicting duplicate-label
# text groups, then keep one representative row per normalized text.
conflict_texts = set(conflict_groups['text_norm'])
base = df[~df['text_norm'].isin(conflict_texts)].copy()
base = base.sort_index().drop_duplicates('text_norm', keep='first')

N_PER_CLASS = 50_000
RANDOM_STATE = 42
available = base['sentiment'].value_counts()
if (available < N_PER_CLASS).any():
    raise ValueError(f'Insufficient unique texts for balanced sample: {available.to_dict()}')

eval_sample = (base.groupby('sentiment', group_keys=False, observed=True)
    .sample(n=N_PER_CLASS, random_state=RANDOM_STATE)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True))

# Preserve raw text for tagging. Create separate evaluation text with source-like emoticons removed.
def strip_source_like_emoticons(text):
    t = ANY_SOURCE_LIKE_RE.sub(' ', str(text))
    return re.sub(r'\s+', ' ', t).strip()

# Keep whitespace normalisation conceptually separate from emoticon removal.
# The v2 audit mistakenly counted ordinary whitespace cleanup as emoticon removal.
eval_sample['text_eval'] = eval_sample['text'].map(strip_source_like_emoticons)
eval_sample['source_like_emoticon_removed'] = eval_sample['has_source_like_emoticon'].astype(bool)
eval_sample['text_whitespace_normalized'] = (
    eval_sample['text'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
    != eval_sample['text'].astype(str)
)

print('Evaluation sample rows:', f'{len(eval_sample):,}')
print(eval_sample['sentiment'].value_counts().to_string())
print('Rows containing/removing source-like emoticons:', int(eval_sample['source_like_emoticon_removed'].sum()))
print('Rows affected by whitespace normalisation:', int(eval_sample['text_whitespace_normalized'].sum()))
print('Empty after stripping:', int((eval_sample['text_eval'].str.strip()=='').sum()))


Evaluation sample rows: 100,000
sentiment
positive    50000
negative    50000
Rows containing/removing source-like emoticons: 864
Rows affected by whitespace normalisation: 99657
Empty after stripping: 0


## Cell 7 — Frozen linguistic-tagging functions applied to raw text


In [8]:
SLANG = list(freeze['slang_heavy']['lexicon'])
SLANG_TOKEN_RE = re.compile(r'(?<!\w)(' + '|'.join(sorted(map(re.escape, SLANG), key=len, reverse=True)) + r')(?!\w)', re.I)
SARCASM_RE = re.compile(freeze['sarcasm_indicated']['regex'], re.I)
EMOJI_THRESHOLD = float(freeze['emoji_heavy']['threshold'])
SLANG_THRESHOLD = float(freeze['slang_heavy']['threshold'])
REF_MIN_WORDS = int(freeze['reference_unmarked']['minimum_word_count'])
MIN_STABLE_N = int(freeze['minimum_n_for_stable_quantitative_interpretation'])

def word_count_raw(t): return len(str(t).split())
def emoji_count_full_sequences(t): return len(emoji.emoji_list(str(t)))
def slang_count(t): return len(SLANG_TOKEN_RE.findall(str(t).lower()))
def sarcasm_indicated(t): return bool(SARCASM_RE.search(str(t).lower()))

x = eval_sample
x['word_count'] = x['text'].map(word_count_raw)
x['emoji_count'] = x['text'].map(emoji_count_full_sequences)
x['slang_count'] = x['text'].map(slang_count)
denom = x['word_count'].clip(lower=1)
x['emoji_density'] = x['emoji_count']/denom
x['slang_ratio'] = x['slang_count']/denom
x['is_emoji_heavy'] = x['emoji_density'] > EMOJI_THRESHOLD
x['is_slang_heavy'] = x['slang_ratio'] > SLANG_THRESHOLD
x['is_sarcasm_indicated'] = x['text'].map(sarcasm_indicated)
x['is_reference_candidate'] = (~x['is_emoji_heavy'] & ~x['is_slang_heavy'] & ~x['is_sarcasm_indicated'] & (x['word_count'] >= REF_MIN_WORDS))

def assign(r):
    if r['is_sarcasm_indicated']: return 'sarcasm-indicated'
    if r['is_emoji_heavy']: return 'emoji-heavy'
    if r['is_slang_heavy']: return 'slang-heavy'
    if r['is_reference_candidate']: return 'reference-unmarked'
    return 'other'
x['subgroup'] = x.apply(assign, axis=1)
print('Frozen rules applied without threshold modification.')


Frozen rules applied without threshold modification.


## Cell 8 — Pre-priority prevalence and overlap


In [10]:
FLAGS=['is_emoji_heavy','is_slang_heavy','is_sarcasm_indicated','is_reference_candidate']
flag_prevalence = pd.DataFrame({
    'flag':FLAGS,
    'count':[int(eval_sample[c].sum()) for c in FLAGS],
    'percentage':[float(eval_sample[c].mean()*100) for c in FLAGS]
})
print(flag_prevalence.to_string(index=False))

focus=['is_emoji_heavy','is_slang_heavy','is_sarcasm_indicated']
overlap_matrix=pd.DataFrame(index=focus, columns=focus, dtype=int)
for a in focus:
    for b in focus:
        overlap_matrix.loc[a,b]=int((eval_sample[a]&eval_sample[b]).sum())
print('Overlap matrix:')
print(overlap_matrix)


                  flag  count  percentage
        is_emoji_heavy     72       0.072
        is_slang_heavy   2801       2.801
  is_sarcasm_indicated    171       0.171
is_reference_candidate  73512      73.512
Overlap matrix:
                      is_emoji_heavy  is_slang_heavy  is_sarcasm_indicated
is_emoji_heavy                  72.0             2.0                   0.0
is_slang_heavy                   2.0          2801.0                   4.0
is_sarcasm_indicated             0.0             4.0                 171.0


## Cell 9 — Final subgroup coverage and binary class structure


In [12]:
subgroup_counts=(eval_sample.groupby('subgroup', observed=True).size().rename('n').reset_index())
subgroup_counts['percentage']=subgroup_counts['n']/len(eval_sample)*100
subgroup_counts['eligible_n_ge_30']=subgroup_counts['n']>=MIN_STABLE_N
subgroup_counts['interpretation']=np.where(subgroup_counts['eligible_n_ge_30'],'quantitative coverage available','descriptive/exploratory only')
print(subgroup_counts.sort_values('n', ascending=False).to_string(index=False))

subgroup_class=(eval_sample.groupby(['subgroup','sentiment'], observed=True).size().rename('count').reset_index())
subgroup_class['within_subgroup_pct']=subgroup_class['count']/subgroup_class.groupby('subgroup', observed=True)['count'].transform('sum')*100
print('Subgroup x sentiment:')
print(subgroup_class.to_string(index=False))


          subgroup     n  percentage  eligible_n_ge_30                  interpretation
reference-unmarked 73512      73.512              True quantitative coverage available
             other 23450      23.450              True quantitative coverage available
       slang-heavy  2795       2.795              True quantitative coverage available
 sarcasm-indicated   171       0.171              True quantitative coverage available
       emoji-heavy    72       0.072              True quantitative coverage available
Subgroup x sentiment:
          subgroup sentiment  count  within_subgroup_pct
       emoji-heavy  negative     27            37.500000
       emoji-heavy  positive     45            62.500000
             other  negative  10713            45.684435
             other  positive  12737            54.315565
reference-unmarked  negative  37688            51.267820
reference-unmarked  positive  35824            48.732180
 sarcasm-indicated  negative     80            46.783626


## Cell 10 — External-role freeze


In [13]:
role_decision={
 'dataset':'Sentiment140',
 'source_dataset_repo':DATASET_REPO,
 'source_task':'binary distant-supervision sentiment classification',
 'role_in_thesis':'binary_external_robustness',
 'used_for_training':False,
 'used_for_model_selection':False,
 'used_for_feature_selection':False,
 'label_space':['negative','positive'],
 'label_space_shift_from_tweeteval':True,
 'evaluation_sample':{
   'rows':int(len(eval_sample)),
   'per_class':N_PER_CLASS,
   'random_state':RANDOM_STATE,
   'unique_normalized_texts':True,
   'conflicting_duplicate_labels_excluded':True
 },
 'evaluation_text_policy':'model evaluation uses text_eval with conservative label-source-like ASCII emoticons removed; linguistic subgroup tagging uses original raw text',
 'linguistic_tagging':{
   'rule_source':'TweetEval frozen configuration',
   'threshold_tuning_on_sentiment140':False,
   'minimum_n_for_stable_quantitative_interpretation':MIN_STABLE_N,
   'pooling_across_datasets_for_eligibility':False
 },
 'interpretation_constraint':'Results are binary robustness evidence only and are not directly comparable to TweetEval three-class macro-F1 without explicit label-space qualification.'
}
print(json.dumps(role_decision, indent=2))


{
  "dataset": "Sentiment140",
  "source_dataset_repo": "contemmcm/sentiment140",
  "source_task": "binary distant-supervision sentiment classification",
  "role_in_thesis": "binary_external_robustness",
  "used_for_training": false,
  "used_for_model_selection": false,
  "used_for_feature_selection": false,
  "label_space": [
    "negative",
    "positive"
  ],
  "label_space_shift_from_tweeteval": true,
  "evaluation_sample": {
    "rows": 100000,
    "per_class": 50000,
    "random_state": 42,
    "unique_normalized_texts": true,
    "conflicting_duplicate_labels_excluded": true
  },
  "evaluation_text_policy": "model evaluation uses text_eval with conservative label-source-like ASCII emoticons removed; linguistic subgroup tagging uses original raw text",
  "linguistic_tagging": {
    "rule_source": "TweetEval frozen configuration",
    "threshold_tuning_on_sentiment140": false,
    "minimum_n_for_stable_quantitative_interpretation": 30,
    "pooling_across_datasets_for_eligibility"

## Cell 11 — Save authoritative outputs


In [14]:
eval_sample.to_parquet(OUTDIR/'df_sentiment140_eval_sample_final.parquet', index=False)
dup_groups.to_csv(OUTDIR/'sentiment140_duplicate_groups.csv', index=False)
conflict_groups.to_csv(OUTDIR/'sentiment140_conflicting_label_duplicates.csv', index=False)
leakage_summary.to_csv(OUTDIR/'sentiment140_source_like_emoticon_audit.csv', index=False)
flag_prevalence.to_csv(OUTDIR/'sentiment140_flag_prevalence.csv', index=False)
overlap_matrix.to_csv(OUTDIR/'sentiment140_overlap_matrix.csv')
subgroup_counts.to_csv(OUTDIR/'sentiment140_subgroup_coverage.csv', index=False)
subgroup_class.to_csv(OUTDIR/'sentiment140_subgroup_class_counts.csv', index=False)

integrity_summary={
 'rows_full':int(len(df)),
 'class_counts':{k:int(v) for k,v in df['sentiment'].value_counts().to_dict().items()},
 'missing_text_rows':missing_text,
 'empty_text_rows':empty_text,
 'unique_normalized_texts':int(df['text_norm'].nunique()),
 'duplicate_normalized_text_groups':int(len(dup_groups)),
 'extra_duplicate_rows':int((dup_groups['row_count']-1).sum()),
 'conflicting_label_duplicate_groups':int(len(conflict_groups)),
 'source_like_emoticon_rows':int(df['has_source_like_emoticon'].sum()),
 'evaluation_sample_rows':int(len(eval_sample)),
 'evaluation_sample_rows_with_source_like_emoticon_removed':int(eval_sample['source_like_emoticon_removed'].sum()),
 'evaluation_sample_rows_with_whitespace_normalized':int(eval_sample['text_whitespace_normalized'].sum()),
 'evaluation_sample_empty_after_stripping':int((eval_sample['text_eval'].str.strip()=='').sum())
}
with open(OUTDIR/'sentiment140_integrity_summary.json','w',encoding='utf-8') as f: json.dump(integrity_summary,f,indent=2)
with open(OUTDIR/'sentiment140_role_freeze.json','w',encoding='utf-8') as f: json.dump(role_decision,f,indent=2)

print('Saved authoritative Sentiment140 outputs to:', OUTDIR)
for p in sorted(OUTDIR.iterdir()): print(' -',p.name)


Saved authoritative Sentiment140 outputs to: /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Final_Thesis/EDA/Results/Sentiment140
 - df_sentiment140_eval_sample_final.parquet
 - sentiment140_conflicting_label_duplicates.csv
 - sentiment140_duplicate_groups.csv
 - sentiment140_flag_prevalence.csv
 - sentiment140_integrity_summary.json
 - sentiment140_overlap_matrix.csv
 - sentiment140_role_freeze.json
 - sentiment140_source_like_emoticon_audit.csv
 - sentiment140_subgroup_class_counts.csv
 - sentiment140_subgroup_coverage.csv


## Cell 12 — Final freeze checklist


In [16]:
print('='*72)
print('SENTIMENT140 EDA v3 — FREEZE CHECKLIST')
print('='*72)
print('Full rows:', f'{len(df):,}')
print('Binary labels:', df['sentiment'].value_counts().to_dict())
print('Conflicting duplicate-label groups:', len(conflict_groups))
print('Evaluation sample:', len(eval_sample), '(50k per class, unique normalized text)')
print('Frozen TweetEval linguistic rules reused:', True)
print('Thresholds tuned on Sentiment140:', False)
print('Used for training/selection:', False)
print('Role:', role_decision['role_in_thesis'])
print('Subgroup coverage:')
print(subgroup_counts.sort_values('n',ascending=False).to_string(index=False))
print('Interpretation: Sentiment140 is retained only as binary external robustness evidence. It is not pooled with other datasets to manufacture subgroup sample size.')


SENTIMENT140 EDA v3 — FREEZE CHECKLIST
Full rows: 1,600,000
Binary labels: {'positive': 800000, 'negative': 800000}
Conflicting duplicate-label groups: 2385
Evaluation sample: 100000 (50k per class, unique normalized text)
Frozen TweetEval linguistic rules reused: True
Thresholds tuned on Sentiment140: False
Used for training/selection: False
Role: binary_external_robustness
Subgroup coverage:
          subgroup     n  percentage  eligible_n_ge_30                  interpretation
reference-unmarked 73512      73.512              True quantitative coverage available
             other 23450      23.450              True quantitative coverage available
       slang-heavy  2795       2.795              True quantitative coverage available
 sarcasm-indicated   171       0.171              True quantitative coverage available
       emoji-heavy    72       0.072              True quantitative coverage available
Interpretation: Sentiment140 is retained only as binary external robustness evide